In [ ]:
from pathlib import Path
from yaml import safe_load
import numpy as np
import geopandas as gp

import yaml
import sys
import subprocess
import datetime
# get the current date and time
now = datetime.datetime.now()

# clawpack 
from clawpack.visclaw import gridtools
from clawpack.pyclaw.solution import Solution

# matplotlib
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap

# Adresse du dossier 
proj_dir        = Path.cwd()
# Chargement du module AVAC
sys.path.insert(0, str(proj_dir / "AVAC"))
from module_avac import reading_raster_file_features, reload_avac, reading_raster_file, \
    export_claw_dem, export_claw_dem_window, determine_file_type, plot_topo
# Chargement du module Waves
from module_waves import reload_wave, format_m
from module_waves import create_boundary_conditions
from module_waves import create_mask
from module_waves import check_domain_in_topo


# latex
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['text.latex.preamble'] = r'\usepackage{libertine}'
plt.rcParams['font.size'] = 12

In [ ]:
# répertoires de travail
avac_output_dir = proj_dir/f"AVAC/_output"
avac_dir        = proj_dir / "AVAC"
BC_dir          = proj_dir / "CL"
topo_dir        = proj_dir / "Topo"
images_dir      = proj_dir / "Figures"
export_dir      = proj_dir / "Topo"
print(f"Répertoire du projet : {proj_dir}")
print(f"- répertoire AVAC : {avac_dir}")
print(f"- répertoire des conditions limites : {BC_dir}")
print(f"- répertoire des fichiers topo : {topo_dir}")
print(f"- répertoire des figures : {images_dir}")
print(f"- répertoire d'export : {export_dir}")
Path(images_dir).mkdir(exist_ok=True)
Path(export_dir).mkdir(exist_ok=True)

# paramètres du calcul
topo_files = {'coarse':'topo2m_c.asc','fine':'topo1m_simple.asc','mask_shp':'masque.shp','mask_raster':'mask.asc','missing_value':-9999}

lake       = {'topography':topo_files['fine'], # clawpack-comptatible topography for the computations
              'water_level':1532.24,
              'xmin':965670,'xmax':965900,'ymin':6536050,'ymax':6536300
              }

computation= {'damping': 0.3, 'mode':'bc','cell_size':.5,'t_0':30,'t_max':120,'nb_simul':180,'output_directory':'_output', \
    'limiter':'mc','track_mass':True,'mass_frac_stop':0.01,'force_stop':False, \
               'mass_threshold_velocity':.01, 'initial_mass':False,
    'cfl_target':0.5,'cfl_max':1,\
               'refinement':1,'max_iter':100000, 'boundary':'user','dry_limit':0.0001}

dict_gauges = {'gauge_recording':False}

rheology    = {'rho':1000,'gravity':9.81,'Strickler':40,'friction':True,'friction_depth_limit':20,'wave_tolerance_flag':0.2}

output      = {'delta_t':.5,'output_format':'binary', # ascii or binary
               'verbosity':0,'Language': 'French'}




date = {'date':now}
# Assemblage des dictionnaires
parameters  = {'topo_files':topo_files,'lake':lake ,'computation':computation,'gauges':dict_gauges, 
               'rheology':rheology, 'output':output, 'date':date}
# Sauvegarde du fichier de configuration
file_name = 'impulse_configuration.yaml'
with open(file_name, 'w') as file:
    yaml.dump(parameters, file)

print(f"Ficher de configuration {file_name} sauvegardé")

 
with open(proj_dir / "impulse_configuration.yaml") as file:
    config = safe_load(file)
    topo = config["topo_files"]
    lake = config["lake"]

with open(avac_dir / "AVAC_configuration.yaml") as file:
    config = safe_load(file)
    avac_output = config["output"]
     

# Jauges

In [ ]:
gauge_recording = True
if gauge_recording:
    # Read the shapefile
    pts = gp.read_file(topo_dir / 'jauges.shp')
    nb_pts = len(pts)
    nb_pts
    dict_gauges = {'gauge_recording':True}
    for i in range(nb_pts):
        x = pts.geometry[i].coords.xy[0][0]
        y = pts.geometry[i].coords.xy[1][0]
        print(f"jauge n° {i+1} : x = {format_m(x)} m et y = {format_m(y)} m")
        dict_gauges[str(i)]={'x':float(x), 'y':float(y)}
else:
    print("pas de jauges !")

In [ ]:
if gauge_recording:
    parameters['gauges'] = dict_gauges
    file_name            = 'impulse_configuration.yaml'
    with open(file_name, 'w') as file:
        yaml.dump(parameters, file)

    print(f"Fichier de configuration {file_name} mis à jour...")

# Création du masque et de la topo_fine

In [ ]:
# Lecture de la topo et vérification de la consistance du domaine de calcul avec la topo dispo
topo_fine = reading_raster_file(str(topo_dir / topo_files['fine']))
xmin, xmax, ymin, ymax, nbx, nby, cell_size, dictionary_domain, failure, remarks, grid_type  = \
        reading_raster_file_features(topo_dir / lake['topography'])

print(f"* xmin = {xmin} m et xmax = {xmax} m ; nbx = {nbx} ; dx = {cell_size} m")
print(f"* ymin = {ymin} m et ymax = {ymax} m ; nby = {nby} ; dy = {cell_size} m")
print(f"* type de grille = {grid_type}  ")
print(f"* format de grille = {determine_file_type(topo_dir / lake['topography'])}  ")


In [ ]:
# Test de compatibilité
print(f"Le fichier topo pour le calcul des vagues est {topo_files['fine']}")
print(f"Ce fichier est de type {determine_file_type(str(topo_dir / topo_files['fine']))}.")
if determine_file_type(str(topo_dir / topo_files['fine'])) != 'claw':
    print(f"Attention : le fichier topographique n'est pas compatible avec geoclaw !")
    print("Conversion vers le format clawpack et transfert vers topography_lake.asc...")
    lake['topography']               = 'topography_lake.asc'
    parameters['lake']['topography'] = 'topography_lake.asc'
    parameters['topo_files']['added_clawpack-compatible_topo'] = 'topography_lake.asc'
    file_name                        = 'impulse_configuration.yaml'
    export_claw_dem(xmin,xmax,ymin,ymax,nbx,nby,topo_fine.Z, name_file=str(topo_dir / 'topography_lake.asc'))
    with open(file_name, 'w') as file:
        yaml.dump(parameters, file)
else:
    print("Le fichier topographique est compatible avec geoclaw. Je le conserve.")
check_domain_in_topo(lake, xmin, xmax, ymin, ymax)

## Export du masque

Rasterisation du shapefile `masque.shp` sur la grille final du lac  
- **0** → intérieur du polygone (lac)  
- **1** → extérieur (zones sèches)

In [ ]:
reload_wave()
from module_waves import create_mask

In [ ]:
# création du masque
create_mask(proj_dir,topo_dir,topo_files,lake,erase=False,mask_cell_size=computation['cell_size'])

In [ ]:
# Ensemble des données topo
mask_file  = reading_raster_file(topo_dir / 'mask.asc')
masked_dem = np.ma.masked_invalid(mask_file.Z)# Mask NaN values
topo_file  = reading_raster_file(topo_dir / lake['topography'])

resampling = True
sampling_factor = 10
from copy import copy
topo_file_resampled = copy(topo_file)
topo_file_resampled.Z = topo_file.Z[::5, ::5]
topo_file_resampled.x = topo_file.x[::5]
topo_file_resampled.y = topo_file.y[::5]

if resampling:
    fig, ax, x0, y0 = plot_topo(topo_file_resampled, step=500)
else:
    fig, ax, x0, y0 = plot_topo(topo_file,step=500)

# figure
ax.add_patch(plt.Rectangle((lake['xmin']-x0,lake['ymin']-y0) , width=lake['xmax']-lake['xmin'], 
                           height=lake['ymax']-lake['ymin'], ls="-", lw=2, ec="red", fc="none"))



cmap_mask = ListedColormap([(1, 1, 0, 1),   # index 0 → jaune opaque
                            (0, 0, 0, 0)])   # index 1 → transparent

plt.imshow(masked_dem, origin='lower',
           extent=[xmin-xmin, xmax-xmin, ymin-ymin, ymax-ymin],
           cmap=cmap_mask, vmin=0, vmax=1)

fig.savefig(images_dir / "vue_densemble.png",dpi = 300, bbox_inches = 'tight')

In [ ]:
fig.savefig(images_dir / "vue_densemble_LR.png",dpi = 100, bbox_inches = 'tight')

In [ ]:
# zoom sur le lac

resampling = False
if resampling:
    fig, ax, x0, y0 = plot_topo(topo_file_resampled, step=50)
else:
    fig, ax, x0, y0 = plot_topo(topo_file,step=50)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=2, ec="red", fc="none"))
 
cmap_mask = ListedColormap([(1, 1, 0, 1), (0, 0, 0, 0)])
plt.imshow(masked_dem, origin='lower',
           extent=[xmin-xmin, xmax-xmin, ymin-ymin, ymax-ymin],
           cmap=cmap_mask, vmin=0, vmax=1)

# Zoom sur le lac avec une marge
margin = 100  # mètres
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)
for k in range(len(dict_gauges)-1):
    x = dict_gauges[str(k)]['x']
    y = dict_gauges[str(k)]['y']
    plt.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)

fig.savefig(images_dir / "vue_rapprochée.png",dpi = 300, bbox_inches = 'tight')


# Création des conditions aux limites

In [ ]:
reload_wave()
from module_waves import create_boundary_conditions
q_dict, flux_dict, times = create_boundary_conditions(proj_dir,lake,BC_dir,damping = computation['damping'])

In [ ]:
# Variation du débit entrant
print(f"Flux entrant dans le domaine de calcul du lac, avec un coefficient de transfert de qdm = {computation['damping']}")
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(9)
fig.set_figwidth(10)
dict_boundary = {'bottom':'(a) sud', 'right':'(b) est', 'top':'(c) nord', 'left':'(d) ouest'}
 
boundaries = list(dict_boundary.keys())
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(times,flux_dict[b])
    #ax.text()
    plt.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$Q$ (m³/s)")

fig.savefig(images_dir / "flux.png",dpi = 300, bbox_inches = 'tight')